In [ ]:
!git clone https://github.com/AI4Bharat/Indic-TTS.git
%cd /content/Indic-TTS/inference

# Define the file path and name
file_path = "/content/Indic-TTS/inference/src/inference.py"

with open(file_path, "w") as file:
    file.truncate(0)  # Clears the file contents

# Write Python code to the file
with open(file_path, "w") as file:
    file.write("""
import io
import re
import base64
import numpy as np
import traceback
from typing import Union

from TTS.utils.synthesizer import Synthesizer
from aksharamukha.transliterate import process as aksharamukha_xlit
from scipy.io.wavfile import write as scipy_wav_write

import nltk
import pysbd

from .models.common import Language
from .models.request import TTSRequest
from .models.response import AudioFile, AudioConfig, TTSResponse, TTSFailureResponse
from .utils.text import TextNormalizer
from .utils.paragraph_handler import ParagraphHandler
from src.postprocessor import PostProcessor

class TextToSpeechEngine:
    def __init__(
        self,
        models: dict,
        allow_transliteration: bool = False,
        enable_denoiser: bool = False,
    ):
        self.models = models
        # TODO: Ability to instantiate models by accepting standard paths or auto-downloading

        code_mixed_found = False
        if allow_transliteration:
            # Initialize Indic-Xlit models for the languages corresponding to TTS models
            from ai4bharat.transliteration import XlitEngine
            xlit_langs = set()

            for lang in list(models):
                if lang == 'en':
                    continue # No need of any Indic-transliteration for English

                if '+' in lang:
                    # If it's a code-mixed model like Hinglish, we need Hindi Xlit for non-English words
                    # TODO: Make it mandatory irrespective of `allow_transliteration` boolean
                    lang = lang.split('+')[1]
                    code_mixed_found = True
                xlit_langs.add(lang)

            self.xlit_engine = XlitEngine(xlit_langs, beam_width=6)
        else:
            self.xlit_engine = None

        self.text_normalizer = TextNormalizer()
        self.paragraph_handler = ParagraphHandler()
        self.sent_seg = pysbd.Segmenter(language="en", clean=True)

        self.orig_sr = 22050 # model.output_sample_rate
        self.enable_denoiser = enable_denoiser
        if enable_denoiser:
            from src.postprocessor import Denoiser
            self.target_sr = 16000
            self.denoiser = Denoiser(self.orig_sr, self.target_sr)
        else:
            self.target_sr = self.orig_sr

        self.post_processor = PostProcessor(self.target_sr)

        if code_mixed_found:
            # Dictionary of English words
            import enchant
            from enchant.tokenize import get_tokenizer

            self.enchant_dicts = {
                "en_US": enchant.Dict("en_US"),
                "en_GB": enchant.Dict("en_GB"),
            }
            self.enchant_tokenizer = get_tokenizer("en")

    def concatenate_chunks(self, wav: np.ndarray, wav_chunk: np.ndarray):
        # TODO: Move to utils
        if type(wav_chunk) != np.ndarray:
            wav_chunk = np.array(wav_chunk)
        if wav is None:
            return wav_chunk
        return np.concatenate([wav, wav_chunk])

    def infer_from_request(
        self,
        request: TTSRequest,
        transliterate_roman_to_native: bool = False
    ) -> TTSResponse:

        config = request.config
        lang = config.language.sourceLanguage
        gender = config.gender

        # If there's no separate English model, use the Hinglish one
        if lang == "en" and lang not in self.models and "en+hi" in self.models:
            lang = "en+hi"

        if lang not in self.models:
            return TTSFailureResponse(status_text="Unsupported language!")

        if lang == "brx" and gender == "male":
            return TTSFailureResponse(status_text="Sorry, `male` speaker not supported for this language!")

        # output_list = []

        audio_config = AudioConfig(language=Language(sourceLanguage=lang))
        return TTSResponse(audio=output_list, config=audio_config)

    def infer_from_text(
        self,
        input_text: str,
        lang: str,
        speaker_name: str,
        transliterate_roman_to_native: bool = True
    ) -> np.ndarray:

        # If there's no separate English model, use the Hinglish one
        if lang == "en" and lang not in self.models and "en+hi" in self.models:
            lang = "en+hi"

        input_text, primary_lang, secondary_lang = self.parse_langs_normalise_text(input_text, lang)

        wav = None
        paragraphs = self.paragraph_handler.split_text(input_text)

        for paragraph in paragraphs:
            paragraph = self.handle_transliteration(paragraph, primary_lang, transliterate_roman_to_native)
            paras = []
            for sent in self.sent_seg.segment(paragraph):
                if sent.strip() and not re.match(r'^[_\W]+$', sent.strip()):
                    paras.append(sent.strip())
            paragraph = " ".join(paras)

            # Run Inference. TODO: Support for batch inference
            wav_chunk = self.models[lang].tts(paragraph, speaker_name=speaker_name, style_wav="")

            wav_chunk = self.postprocess_audio(wav_chunk, primary_lang, speaker_name)
            # Concatenate current chunk with previous audio outputs
            wav = self.concatenate_chunks(wav, wav_chunk)
        return wav

    def parse_langs_normalise_text(self, input_text: str, lang: str) -> Union[str, str, str]:
        # If there's no separate English model, use the Hinglish one if present
        if lang == "en" and lang not in self.models and "en+hi" in self.models:
            lang = "en+hi"

        if lang == "en+hi": # Hinglish (English+Hindi code-mixed)
            primary_lang, secondary_lang = lang.split('+')
        else:
            primary_lang = lang
            secondary_lang = None

        input_text = self.text_normalizer.normalize_text(input_text, primary_lang)
        if secondary_lang:
            # TODO: Write a proper `transliterate_native_words_using_eng_dictionary`
            input_text = self.transliterate_native_words_using_spell_checker(input_text, secondary_lang)

        return input_text, primary_lang, secondary_lang

    def handle_transliteration(self, input_text: str, primary_lang: str, transliterate_roman_to_native: bool) -> str:
        if transliterate_roman_to_native and primary_lang != 'en':
            input_text = self.transliterate_sentence(input_text, primary_lang)

            # Manipuri was trained using the Central-govt's Bangla script
            # So convert the words in native state-govt script to Eastern-Nagari
            if primary_lang == "mni":
                # TODO: Delete explicit-schwa
                input_text = aksharamukha_xlit("MeeteiMayek", "Bengali", input_text)
        return input_text

    def preprocess_text(
        self,
        input_text: str,
        lang: str,
        # speaker_name: str,
        transliterate_roman_to_native: bool = True
    ) -> np.ndarray:

        input_text, primary_lang, secondary_lang = self.parse_langs_normalise_text(input_text, lang)
        input_text = self.handle_transliteration(input_text, primary_lang, transliterate_roman_to_native)
        return input_text

    def postprocess_audio(self, wav_chunk, primary_lang, speaker_name):
        if self.enable_denoiser:
            wav_chunk = self.denoiser.denoise(wav_chunk)
        wav_chunk = self.post_processor.process(wav_chunk, primary_lang, speaker_name)
        return wav_chunk

    def transliterate_native_words_using_spell_checker(self, input_text, lang):
        tokens = [result[0] for result in self.enchant_tokenizer(input_text)]
        pos_tags = [result[1] for result in nltk.tag.pos_tag(tokens)]

        # Transliterate non-English Roman words to Indic
        for word, pos_tag in zip(tokens, pos_tags):
            if pos_tag == "NNP" or pos_tag == "NNPS":
                # Enchant has many proper-nouns as well in its dictionary, don't know why.
                # So if it's a proper-noun, always nativize
                # FIXME: But NLTK's `averaged_perceptron_tagger` does not seem to be 100% accurate, it has false positives 🤦‍♂️
                pass
            elif self.enchant_dicts["en_US"].check(word) or self.enchant_dicts["en_GB"].check(word):
                # TODO: Merge British and American dicts into 1 somehow
                continue

            # Convert "Ram's" -> "Ram". TODO: Think what are the failure cases
            word = word.split("'")[0]

            transliterated_word = self.transliterate_sentence(word, lang)
            input_text = input_text.replace(word, transliterated_word, 1)
        return input_text

    def transliterate_sentence(self, input_text, lang):
        if not self.xlit_engine:
            return input_text

        if lang == "raj":
            lang = "hi" # Approximate

        return self.xlit_engine.translit_sentence(input_text, lang)
""")
 print("Python code has been written to:", file_path)

!pip install jupyter librosa pandas scipy pytorch_lightning scikit-learn seaborn soundfile tqdm TTS tensorboard wandb aksharamukha ffmpeg-python ffmpeg num2words translators pyenchant nltk pynini==2.1.5 sacremoses editdistance indic_numtowords cdifflib
!git clone https://github.com/AI4Bharat/indic-numtowords.git
!git clone https://github.com/NVIDIA/NeMo-text-processing.git
%cd /content/Indic-TTS/inference/NeMo-text-processing/

import os
import urllib.request
from zipfile import ZipFile
from urllib.error import HTTPError

# Define the list of URLs to download
file_urls = [
    'https://github.com/AI4Bharat/Indic-TTS/releases/download/v1-checkpoints-release/bn.zip',
    'https://github.com/AI4Bharat/Indic-TTS/releases/download/v1-checkpoints-release/gu.zip',
    'https://github.com/AI4Bharat/Indic-TTS/releases/download/v1-checkpoints-release/hi.zip',
    'https://github.com/AI4Bharat/Indic-TTS/releases/download/v1-checkpoints-release/kn.zip',
    'https://github.com/AI4Bharat/Indic-TTS/releases/download/v1-checkpoints-release/ml.zip',
    'https://github.com/AI4Bharat/Indic-TTS/releases/download/v1-checkpoints-release/mr.zip',
    'https://github.com/AI4Bharat/Indic-TTS/releases/download/v1-checkpoints-release/ta.zip',
    'https://github.com/AI4Bharat/Indic-TTS/releases/download/v1-checkpoints-release/te.zip'
]

# Create a folder to store the downloaded files
download_folder = 'downloads'
os.makedirs(download_folder, exist_ok=True)

# Download each file individually
for file_url in file_urls:
    file_name = os.path.basename(file_url)
    file_path = os.path.join(download_folder, file_name)

    try:
        urllib.request.urlretrieve(file_url, file_path)
        print(f"Downloaded: {file_name} to {file_path}")
    except HTTPError as e:
        print(f"Error downloading {file_name}. HTTP Error {e.code}: {e.reason}")

# Create a folder named 'checkpoints' to store the unzipped contents
checkpoints_folder = 'checkpoints'
os.makedirs(checkpoints_folder, exist_ok=True)

# Create a folder named 'models' inside 'checkpoints'
model_folder = '/content/Indic-TTS/inference/NeMo-text-processing/'
models_folder = os.path.join(model_folder, 'models')
os.makedirs(models_folder, exist_ok=True)

# Create a folder named 'v1' inside 'models'
v1_folder = os.path.join(models_folder, 'v1')
os.makedirs(v1_folder, exist_ok=True)

# Unzip each downloaded file into both the 'checkpoints' folder and 'v1' folder
for file_name in os.listdir(download_folder):
    file_path = os.path.join(download_folder, file_name)

    with ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(checkpoints_folder)
        zip_ref.extractall(v1_folder)

# Print the contents of the 'checkpoints' folder
checkpoint_contents = os.listdir(checkpoints_folder)
print("Files extracted to 'checkpoints' folder:", checkpoint_contents)

# Clean up: Remove the downloaded files
for file_name in os.listdir(download_folder):
    file_path = os.path.join(download_folder, file_name)
    os.remove(file_path)

%cd /content/Indic-TTS/inference
import io
from TTS.utils.synthesizer import Synthesizer
%cd /content/Indic-TTS/inference/NeMo-text-processing/nemo_text_processing
%cd /content/Indic-TTS/inference
%cd /content/Indic-TTS/inference/NeMo-text-processing
from src.inference import TextToSpeechEngine
!pip install python-docx pydub


In [ ]:
import docx
import os
from scipy.io.wavfile import write as scipy_wav_write
import base64
import numpy as np
from scipy.io import wavfile
from pydub import AudioSegment
from src.models.response import AudioFile, AudioConfig, TTSResponse, TTSFailureResponse

def extract_text_from_docx(docx_path):
    doc = docx.Document(docx_path)
    text = ""
    for paragraph in doc.paragraphs:
        text += paragraph.text + "\n"
    return text
docx_path = '/content/Gujarati_transliterated_docx.docx'
extracted_text = extract_text_from_docx(docx_path)

# Initialize Language model

lang = input("Enter the language: ")
gender = input("Enter speaker gender: ")
lang_model  = Synthesizer(
    tts_checkpoint=f'checkpoints/{lang}/fastpitch/best_model.pth',
    tts_config_path=f'checkpoints/{lang}/fastpitch/config.json',
    tts_speakers_file=f'checkpoints/{lang}/fastpitch/speakers.pth',
    tts_languages_file=None,
    vocoder_checkpoint=f'checkpoints/{lang}/hifigan/best_model.pth',
    vocoder_config=f'checkpoints/{lang}/hifigan/config.json',
    encoder_checkpoint="",
    encoder_config="",
    use_cuda=False,
)

# Setup TTS Engine

models = {
    lang: lang_model,
}
engine = TextToSpeechEngine(models)

# TTS inference

raw_audio = engine.infer_from_text(
    input_text = extracted_text,
    lang = lang,
    speaker_name = gender
)
sampling_rate = 22050
# Convert PCM to WAV
byte_io = io.BytesIO()
scipy_wav_write(byte_io, sampling_rate, raw_audio)

with open("audio.wav", "wb") as f:
    f.write(byte_io.read())

# Load the floating-point audio file
float_audio_file = "audio.wav"
sample_rate, float_audio = wavfile.read(float_audio_file)

# Convert float audio to integer PCM (16-bit)
pcm_audio = np.int16(float_audio * 32767)

# Save the PCM audio as WAV
wavfile.write("output_pcm.wav", sample_rate, pcm_audio)

# Load the WAV file
sound = AudioSegment.from_wav("output_pcm.wav")

# Export the sound as MP3
sound.export("output_audio.mp3", format="mp3")

print("Combined audio saved to final_output_audio")